# Assignment 18 - Text Vectorization Techniques

## Objective

In this assignment, I explored different techniques used to convert text into numerical features that machine learning algorithms can understand.

The focus of this assignment is feature extraction only, so no machine learning models are trained. I implemented and compared:

- Manual One-Hot Encoding
- One-Hot Encoding using CountVectorizer
- Bag of Words (BoW)
- N-Grams
- TF-IDF Vectorization

Throughout the notebook, I compare how each technique represents text and discuss the advantages and limitations of each method.

---

## Dataset

**Dataset:** SMS Spam Collection Dataset

**Source:** Kaggle

https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset

For consistency, I am using the same cleaned dataset that I created in Assignment 17. The `final_clean_text` column generated in the previous assignment will be used for all vectorization tasks.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df=pd.read_csv("spam.csv",encoding='latin-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [3]:
df=df.drop(columns=["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"])
df.columns=["label","text"]
df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
import re

In [6]:
def remove_repeated_characters(text):
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

In [7]:
slang_dict = {
    "u": "you",
    "ur": "your",
    "pls": "please",
    "msg": "message",
    "luv": "love",
    "thx": "thanks",
    "thanx": "thanks",
    "tmr": "tomorrow",
    "gr8": "great",
    "wat": "what",
    "wif": "with",
    "gud": "good",
    "nite": "night",
    "coz": "because",
    "c": "see",
    "txt": "text",
    "wk": "week",
    "wkly": "weekly",
    "mins": "minutes",
    "ya": "yeah",
    "yae": "yeah",
    "lor": "lord",
    "da": "the",
    "oso": "also"
}

In [8]:
def replace_slang(text):
    words = text.split()
    updated_words = []
    for word in words:
        if word in slang_dict:
            updated_words.append(slang_dict[word])
        else:
            updated_words.append(word)
    return " ".join(updated_words)

In [13]:
import nltk
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

In [14]:
def remove_stopwords(text):
    words = text.split()
    filtered_words = []
    for word in words:
        if word not in stop_words:
            filtered_words.append(word)
    return ' '.join(filtered_words)

In [9]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [11]:
from nltk.tokenize import word_tokenize
def lemmatize_text(text):
    words = word_tokenize(text)
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(lemmatized_words)

In [15]:
def nlp_preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\S+@\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = remove_repeated_characters(text)
    text = replace_slang(text)
    text = remove_stopwords(text)
    text = lemmatize_text(text)
    return text

In [16]:
df["final_clean_text"] = df["text"].apply(nlp_preprocess)

In [19]:
df.shape
df.info()
df[["label", "text", "final_clean_text"]].head()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   label             5572 non-null   str  
 1   text              5572 non-null   str  
 2   final_clean_text  5572 non-null   str  
dtypes: str(3)
memory usage: 855.4 KB


,label,text,final_clean_text
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,ham,Ok lar... Joking wif u oni...,ok lar joking oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry weekly comp win fa cup final tkts s...
3,ham,U dun say so early hor... U c already then say...,dun say early hor see already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah dont think go usf life around though


In [20]:
df.isnull().sum()

label               0
text                0
final_clean_text    0
dtype: int64

In [22]:
df['final_clean_text'].sample(5)

1236                                         much getting
2095                                   probably want pick
3774    hi spoke maneesha v wed like know satisfied ex...
4049                           ok ok take care understand
1163    new theory argument win situation loses person...
Name: final_clean_text, dtype: str

## Initial Observations

Before applying vectorization, I checked the cleaned text generated in the previous assignment. The messages are already lowercased, punctuation has been removed, stopwords have been filtered out, and the text has been lemmatized. This allows the vectorizers to focus on meaningful words instead of unnecessary noise.

# Part 1 - One-Hot Encoding

One-Hot Encoding represents each unique word as a binary feature. A value of **1** means the word is present in a sentence, while **0** means it is is absent.

To understand how it works, I first implemented One-Hot Encoding manually before using Scikit-learn.

## Task-1

In [25]:
sample_sentences = df["final_clean_text"].head(5)

for i, sentence in enumerate(sample_sentences):
    print(f"Sentence {i+1}:")
    print(sentence)
    print()

Sentence 1:
go jurong point crazy available bugis n great world la e buffet cine got amore

Sentence 2:
ok lar joking oni

Sentence 3:
free entry weekly comp win fa cup final tkts st may text fa receive entry questionstd text ratetcs apply over

Sentence 4:
dun say early hor see already say

Sentence 5:
nah dont think go usf life around though



In [26]:
vocabulary = []
for sentence in sample_sentences:
    words = sentence.split()
    for word in words:
        if word not in vocabulary:
            vocabulary.append(word)

print("Vocabulary Size:", len(vocabulary))
print(vocabulary)

Vocabulary Size: 49
['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'ok', 'lar', 'joking', 'oni', 'free', 'entry', 'weekly', 'comp', 'win', 'fa', 'cup', 'final', 'tkts', 'st', 'may', 'text', 'receive', 'questionstd', 'ratetcs', 'apply', 'over', 'dun', 'say', 'early', 'hor', 'see', 'already', 'nah', 'dont', 'think', 'usf', 'life', 'around', 'though']


In [27]:
encoded_vectors=[]
for sentence in sample_sentences:
    words = sentence.split()
    vector = []
    for vocab_word in vocabulary:
        if vocab_word in words:
            vector.append(1)
        else:
            vector.append(0)
    encoded_vectors.append(vector)

In [28]:
one_hot_df=pd.DataFrame(encoded_vectors, columns=vocabulary)
one_hot_df.index = [f"Sentence {i+1}" for i in range(len(one_hot_df))]
one_hot_df

,go,jurong,point,crazy,available,bugis,n,great,world,la,e,buffet,cine,got,amore,ok,lar,joking,oni,free,entry,weekly,comp,win,fa,cup,final,tkts,st,may,text,receive,questionstd,ratetcs,apply,over,dun,say,early,hor,see,already,nah,dont,think,usf,life,around,though
Sentence 1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Sentence 2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Sentence 3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
Sentence 4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,0,0,0,0,0,0,0
Sentence 5,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1


### Observations

After selecting the first five cleaned messages, I manually created a vocabulary containing **49 unique words**. Each sentence was then represented as a binary vector where:

- **1** indicates that the word is present in the sentence.
- **0** indicates that the word is absent.

While creating the vectors manually helped me understand how One-Hot Encoding works internally, I also noticed that the number of columns grows quickly as the vocabulary increases. Even with only five messages, the encoding already required 49 different features.

## Task 2 - One-Hot Encoding using Scikit-learn

After implementing One-Hot Encoding manually, I used Scikit-learn's `CountVectorizer` with `binary=True`. This produces the same binary representation while automatically building the vocabulary and encoding the sentences.

In [29]:
vectorizer = CountVectorizer(binary=True)
one_hot_matrix = vectorizer.fit_transform(sample_sentences)

In [30]:
vocabulary = vectorizer.get_feature_names_out()
print("Vocabulary Size:", len(vocabulary))
print(vocabulary)

Vocabulary Size: 47
['already' 'amore' 'apply' 'around' 'available' 'buffet' 'bugis' 'cine'
 'comp' 'crazy' 'cup' 'dont' 'dun' 'early' 'entry' 'fa' 'final' 'free'
 'go' 'got' 'great' 'hor' 'joking' 'jurong' 'la' 'lar' 'life' 'may' 'nah'
 'ok' 'oni' 'over' 'point' 'questionstd' 'ratetcs' 'receive' 'say' 'see'
 'st' 'text' 'think' 'though' 'tkts' 'usf' 'weekly' 'win' 'world']


In [31]:
one_hot_sklearn = pd.DataFrame(
    one_hot_matrix.toarray(),
    columns=vocabulary,
    index=[f"Sentence {i+1}" for i in range(len(sample_sentences))]
)

one_hot_sklearn

,already,amore,apply,around,available,buffet,bugis,cine,comp,crazy,cup,dont,dun,early,entry,fa,final,free,go,got,great,hor,joking,jurong,la,lar,life,may,nah,ok,oni,over,point,questionstd,ratetcs,receive,say,see,st,text,think,though,tkts,usf,weekly,win,world
Sentence 1,0,1,0,0,1,1,1,1,0,1,0,0,0,0,0,0,0,0,1,1,1,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
Sentence 2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Sentence 3,0,0,1,0,0,0,0,0,1,0,1,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,1,1,0,0,1,1,0,0,1,0,1,1,0
Sentence 4,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0
Sentence 5,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0


In [32]:
print("Manual Shape :", one_hot_df.shape)
print("Scikit-learn Shape :", one_hot_sklearn.shape)

Manual Shape : (5, 49)
Scikit-learn Shape : (5, 47)


### Observations

The Scikit-learn implementation produced a vocabulary of **47 words**, while my manual implementation produced **49 words**.

The difference occurred because `CountVectorizer` ignores single-character tokens by default. In my manually created vocabulary, words like **"n"** and **"e"** were included, but Scikit-learn automatically removed them using its default tokenization rules.

Although the vocabulary sizes are slightly different, both methods follow the same basic idea of One-Hot Encoding by representing each sentence as a binary vector indicating the presence or absence of each word.

Using Scikit-learn is much faster and more practical, especially for larger datasets where creating the vectors manually would not be efficient.

# Part 2 - Bag of Words (BoW)

Unlike One-Hot Encoding, Bag of Words stores the frequency of each word instead of only indicating whether the word exists or not. This provides more information about how often words appear in each document.

In [35]:
bow = CountVectorizer()
bow_matrix = bow.fit_transform(df["final_clean_text"])

In [36]:
print("Vocabulary Size:", len(bow.vocabulary_))
print("BOW Matrix Shape:", bow_matrix.shape)

Vocabulary Size: 7805
BOW Matrix Shape: (5572, 7805)


In [37]:
vocabulary = bow.get_feature_names_out()
print("First 30 words in the vocabulary:", vocabulary[:30])

First 30 words in the vocabulary: ['__' 'aa' 'aah' 'aaniye' 'aaooright' 'aathilove' 'aathiwhere' 'ab'
 'abbey' 'abdomen' 'abeg' 'abel' 'aberdeen' 'abi' 'ability' 'abiola' 'abj'
 'able' 'abnormally' 'aboutas' 'abroad' 'absence' 'absolutely' 'abstract'
 'abt' 'abta' 'aburo' 'abuse' 'abuser' 'ac']


In [38]:
bow_df = pd.DataFrame(
    bow_matrix[:5].toarray(),
    columns=vocabulary
)

bow_df.iloc[:, :20]

,__,aa,aah,aaniye,aaooright,aathilove,aathiwhere,ab,abbey,abdomen,abeg,abel,aberdeen,abi,ability,abiola,abj,able,abnormally,aboutas
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### Observation

This time I fitted CountVectorizer on the entire dataset instead of only a few sample messages.

The vocabulary became much larger because it now includes unique words from all 5,572 SMS messages. Each row represents one SMS, while each column represents a word from the vocabulary. The values indicate how many times each word appears in a message.

Compared to One-Hot Encoding, Bag of Words stores word frequencies instead of only indicating whether a word is present.